# Анализ облигаций — ежемесячный купон

Данные всегда берутся живыми с API Т-Банка при запуске (тикеры + детали по каждой бумаге) —
никаких сохранённых CSV, поэтому не бывает рассинхрона между старыми и новыми данными.
Каждый запуск занимает ~1–1.5 часа (детали по ~1400+ бумагам).

## Настройки

Всё, что определяет стратегию отбора и вид отчёта — здесь. Поменяй и перезапусти ячейки ниже.

In [ ]:
# === Стратегия отбора облигаций ===
MIN_COUPON_QTY_PER_YEAR = 12       # только ежемесячный купон (12 выплат в год)
MIN_COUPON_TO_PRICE_PCT = 0.02     # купон >= 2% от текущей цены
MIN_PRICE_TO_FACE_PCT = 0.97       # цена >= 97% от номинала
EXCLUDE_QUAL_INVESTOR = True       # исключить бумаги для квалифицированных инвесторов
EXCLUDE_FLOATING_COUPON = True     # исключить облигации с плавающим купоном
EXCLUDE_SECURITIZED = True         # исключить облигации с залоговым обеспечением (СФО)
EXCLUDE_PREMIUM_NEAR_MATURITY = True   # исключить "цена выше номинала при близком погашении"
PREMIUM_NEAR_MATURITY_MAX_YEARS = 0.5  # порог "близкого" погашения, лет

# === Настройки отчёта ===
TOP_N = 10                        # сколько лучших облигаций показывать в каждой группе оценки
MIN_BONDS_PER_SECTOR_CHART = 2    # мин. кол-во бумаг в секторе, чтобы попасть в график среднего купона

## Загрузка данных

Список облигаций и детали по каждой — напрямую у API Т-Банка.

In [ ]:
import fetch_bonds, fetch_bonds_detail, filter_bonds

print("Забираю список облигаций с биржи...")
tickers_df = fetch_bonds.fetch_tickers()
print(f"Тикеров: {len(tickers_df)}")

print("Забираю детали по каждой облигации (~1–1.5 часа)...")
detail_df = fetch_bonds_detail.fetch_details(tickers_df["ticker"].tolist())
print(f"Получено деталей: {len(detail_df)}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
import warnings
warnings.filterwarnings("ignore")

plt.rcParams["figure.dpi"] = 120
plt.rcParams["font.family"] = "DejaVu Sans"
sns.set_theme(style="whitegrid")

df = filter_bonds.apply_filter(
    detail_df,
    min_coupon_qty_per_year=MIN_COUPON_QTY_PER_YEAR,
    min_coupon_to_price_pct=MIN_COUPON_TO_PRICE_PCT,
    min_price_to_face_pct=MIN_PRICE_TO_FACE_PCT,
    exclude_qual_investor=EXCLUDE_QUAL_INVESTOR,
    exclude_floating_coupon=EXCLUDE_FLOATING_COUPON,
)
df = filter_bonds.add_securitization(df)

# типизация
numeric = [
    "price_buy", "price_sell", "price_last", "price_close", "price_pct",
    "face_value", "nkd", "coupon_value", "coupon_pct", "coupon_period_days",
    "coupon_qty_per_year", "yield_to_maturity", "yield_to_client", "total_yield",
    "duration", "rate", "risk_category", "earnings_abs", "earnings_rel",
    "hist_price_1y", "lot_size",
]
for col in numeric:
    df[col] = pd.to_numeric(df[col], errors="coerce")

date_cols = ["next_coupon_date", "maturity_date", "date_to_client"]
for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce", utc=True)

df["maturity_years"] = (df["maturity_date"] - pd.Timestamp.now(tz="UTC")).dt.days / 365
df["coupon_to_price_pct"] = df["coupon_value"] / df["price_last"] * 100
df["annual_coupon"] = df["coupon_value"] * df["coupon_qty_per_year"]

RATE_LABEL = {0: "Низкая", 1: "Средняя", 2: "Высокая"}
df["rate_label"] = df["rate"].map(RATE_LABEL)

# доп. фильтр: цена выше номинала при близком погашении — ловушка (потеряешь на погашении по номиналу)
before = len(df)
if EXCLUDE_PREMIUM_NEAR_MATURITY:
    df = df[~((df["price_last"] > df["face_value"]) & (df["maturity_years"] <= PREMIUM_NEAR_MATURITY_MAX_YEARS))]

# исключаем облигации с залоговым обеспечением (секьюритизация/СФО)
before_sec = len(df)
if EXCLUDE_SECURITIZED:
    df = df[~df["securitization"]]
print(f"Исключено залоговых (секьюритизация/СФО): {before_sec - len(df)}")
print(f"Всего облигаций: {len(df)} (отфильтровано доп.: {before - len(df)} — цена > номинал и срок ≤ {PREMIUM_NEAR_MATURITY_MAX_YEARS} года)")

df.groupby("rate_label")[["coupon_to_price_pct", "duration", "maturity_years"]].describe().round(2)

## Топ по купону/цена — по группам оценки

In [ ]:
COLS_SHOW = ["ticker", "name", "sector", "coupon_to_price_pct",
             "annual_coupon", "price_last", "face_value", "maturity_years", "duration"]

for rate_val, label in [(2, "Высокая"), (1, "Средняя"), (0, "Низкая")]:
    top = (
        df[df["rate"] == rate_val]
        .sort_values("coupon_to_price_pct", ascending=False)
        .head(TOP_N)[COLS_SHOW]
        .reset_index(drop=True)
    )
    top.index += 1
    print(f"\n{'='*60}")
    print(f"  Оценка: {label} ({len(df[df['rate']==rate_val])} облигаций в группе)")
    print(f"{'='*60}")
    display(top.style
        .format({
            "coupon_to_price_pct": "{:.2f}%",
            "annual_coupon": "{:.2f} \u20bd",
            "price_last": "{:.2f} \u20bd",
            "face_value": "{:.0f} \u20bd",
            "maturity_years": "{:.1f} лет",
            "duration": "{:.2f}",
        })
        .background_gradient(subset=["coupon_to_price_pct"], cmap="YlGn")
        .set_caption(f"Топ-{TOP_N} по купон/цена — оценка «{label}»")
    )

## Аналитика

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))
fig.suptitle("Аналитика по облигациям (ежемесячный купон)", fontsize=15, fontweight="bold")

colors = {"Высокая": "#2ecc71", "Средняя": "#f39c12", "Низкая": "#e74c3c"}
order = ["Высокая", "Средняя", "Низкая"]

# 1. Распределение купон/цена по группам
ax = axes[0, 0]
for label in order:
    sub = df[df["rate_label"] == label]["coupon_to_price_pct"].dropna()
    ax.hist(sub, bins=15, alpha=0.7, label=label, color=colors[label])
ax.set_title("Распределение купон/цена за выплату")
ax.set_xlabel("Купон / цена, %")
ax.set_ylabel("Кол-во облигаций")
ax.legend()

# 2. Boxplot купон/цена по оценкам
ax = axes[0, 1]
data_box = [df[df["rate_label"] == l]["coupon_to_price_pct"].dropna() for l in order]
bp = ax.boxplot(data_box, patch_artist=True, tick_labels=order)
for patch, label in zip(bp["boxes"], order):
    patch.set_facecolor(colors[label])
ax.set_title("Купон/цена по группам оценки")
ax.set_ylabel("Купон / цена, %")

# 3. Scatter: дюрация vs купон/цена
ax = axes[0, 2]
for label in order:
    sub = df[df["rate_label"] == label]
    ax.scatter(sub["duration"], sub["coupon_to_price_pct"], alpha=0.7,
               label=label, color=colors[label], s=50)
ax.set_title("Дюрация vs Купон/цена")
ax.set_xlabel("Дюрация (лет)")
ax.set_ylabel("Купон / цена, %")
ax.legend()

# 4. Топ секторов
ax = axes[1, 0]
sector_counts = df["sector"].value_counts().head(10)
sector_counts.plot(kind="barh", ax=ax, color="#3498db")
ax.set_title("Топ секторов")
ax.set_xlabel("Кол-во облигаций")
ax.invert_yaxis()

# 5. Срок до погашения
ax = axes[1, 1]
for label in order:
    sub = df[df["rate_label"] == label]["maturity_years"].dropna()
    ax.hist(sub, bins=12, alpha=0.7, label=label, color=colors[label])
ax.set_title("Распределение по сроку до погашения")
ax.set_xlabel("Лет до погашения")
ax.set_ylabel("Кол-во облигаций")
ax.legend()

# 6. Годовой купон (₽) vs купон/цена
ax = axes[1, 2]
for label in order:
    sub = df[df["rate_label"] == label]
    ax.scatter(sub["annual_coupon"], sub["coupon_to_price_pct"],
               alpha=0.7, label=label, color=colors[label], s=50)
ax.set_title("Годовой купон (₽) vs Купон/цена")
ax.set_xlabel("Годовой купон, ₽")
ax.set_ylabel("Купон / цена, %")
ax.legend()

plt.tight_layout()
plt.show()

## Дополнительная аналитика

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle("Дополнительная аналитика", fontsize=13, fontweight="bold")

# 1. Средний купон/цена по секторам
ax = axes[0]
sector_yield = (
    df.groupby("sector")["coupon_to_price_pct"]
    .agg(["mean", "count"])
    .query("count >= @MIN_BONDS_PER_SECTOR_CHART")
    .sort_values("mean", ascending=True)
)
bars = ax.barh(sector_yield.index, sector_yield["mean"], color="#9b59b6")
ax.set_title("Средний купон/цена по секторам")
ax.set_xlabel("Купон / цена, %")
for bar, (_, row) in zip(bars, sector_yield.iterrows()):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f"{row['mean']:.2f}% (n={int(row['count'])})", va="center", fontsize=8)

# 2. Цена в % от номинала
ax = axes[1]
df["price_to_face"] = df["price_last"] / df["face_value"] * 100
for label in order:
    sub = df[df["rate_label"] == label]["price_to_face"].dropna()
    ax.hist(sub, bins=15, alpha=0.7, label=label, color=colors[label])
ax.axvline(100, color="black", linestyle="--", linewidth=1, label="Номинал")
ax.set_title("Цена в % от номинала")
ax.set_xlabel("Цена / номинал, %")
ax.set_ylabel("Кол-во облигаций")
ax.legend(fontsize=8)

# 3. Срок до погашения vs купон/цена
ax = axes[2]
for label in order:
    sub = df[df["rate_label"] == label]
    ax.scatter(sub["maturity_years"], sub["coupon_to_price_pct"],
               alpha=0.7, label=label, color=colors[label], s=50)
ax.set_title("Срок до погашения vs Купон/цена")
ax.set_xlabel("Лет до погашения")
ax.set_ylabel("Купон / цена, %")
ax.legend()

plt.tight_layout()
plt.show()